In [1]:
#from pyaedt.hfss import Hfss
from ansys.aedt.core import Hfss
import os

solution_type = "Modal" # "Eigenmode", "Modal"

use_antenna = False
use_second_chip = False
do_chip_angle_sweep = False
do_chip_gap_sweep = True
do_chip_width_sweep = False
do_chip_height_sweep = False
do_outer_radius_sweep = False
do_outer_height_sweep = False
do_coax_height_sweep = False

hfss = Hfss(
    projectname="test.aedt",
    designname="A1",
    solution_type=solution_type
)

PyAEDT WARNING: Argument `designname` is deprecated for method `__init__`; use `design` instead.
PyAEDT WARNING: Argument `projectname` is deprecated for method `__init__`; use `project` instead.
PyAEDT INFO: Python version 3.11.13 | packaged by conda-forge | (main, Jun  4 2025, 14:39:58) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.17.4.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\ymino\AppData\Local\Temp\15\pyaedt_ymino_6574bc57-103f-4d09-a063-d40f4743a5fd.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Launching PyAEDT with gRPC plugin.
PyAEDT ERROR: A(n) <class 'psutil.AccessDenied'> error occurred while retrieving information for the active AEDT sessions: (pid=11984, name='ansysedt.exe')
PyAEDT ERROR: A(n) <class 'psutil.AccessDenied'> error occurred while retrieving information for the active AEDT s

In [2]:
hfss["$outer_radius"] = "8mm" # 5mm, 8mm (for 10GHz cut-off)
hfss["$outer_height"] = "38mm" # 38mm
hfss["$coax_radius"] = "1.5mm" # "1.5mm"
hfss["$coax_height"] = "9mm"
hfss["$antenna_height"] = "18mm"
hfss["antenna_radius"] = "1.5mm"
hfss["$antenna_outer_height"] = "8mm"
hfss["antenna_outer_radius"] = "3mm"
hfss["$chip_gap"] = "100um" # 20um
hfss["$chip_width"] = "250um" # 250um
hfss["$chip_height"] = "400um" # 400um (For chip_gap = 20um, galvanic with 3240 ( ~4500 with 45 degree ? ) )
hfss["$chip_theta"] = "45deg"
hfss["$chip_pos_z"] = "0.5*$coax_height"
hfss["$chip_pos_y"] = "$coax_radius + 0.5*($outer_radius - $coax_radius)"

hfss["box_height"] = "40mm"
hfss["$box_length"] = "2*$outer_radius + 2mm" #"15mm"

In [3]:
box_config = dict(
    position = ["-$box_length/2", "-$box_length/2", "$outer_height - box_height"],
    dimensions_list = ["$box_length", "$box_length", "box_height"],
    name = "box",
    matname = "copper"
)
if use_antenna:
    box_config.update( dimensions_list = ["$box_length", "$box_length", "box_height + $antenna_outer_height"] )

box_object = hfss.modeler.create_box( **box_config )

## Make Vacuum object
#_, vacuum_name = hfss.modeler.duplicate_around_axis(box_object, "Z", angle = 0)
vacuum_object = box_object.clone()
vacuum_object.name = "boundary"
vacuum_object = hfss.modeler.get_object_from_name("boundary")
hfss.assign_material("boundary", "vacuum")
if solution_type == "Eigenmode":
    hfss.modeler.move_face([vacuum_object.top_face_z], offset=10) # offset in mm

PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT WARNING: Argument `position` is deprecated for method `create_box`; use `origin` instead.
PyAEDT WARNING: Argument `dimensions_list` is deprecated for method `create_box`; use `sizes` instead.
PyAEDT WARNING: Argument `matname` is deprecated for method `create_box`; use `material` instead.
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec


In [4]:
cylinder_config = dict(
    cs_axis="Z" ,
    position=[0, 0, 0], 
    radius="$outer_radius", 
    height="$outer_height", 
    name="outer", 
    matname="vacuum"
)

cylinder_object = hfss.modeler.create_cylinder( **cylinder_config )

PyAEDT WARNING: Argument `cs_axis` is deprecated for method `create_cylinder`; use `orientation` instead.
PyAEDT WARNING: Argument `position` is deprecated for method `create_cylinder`; use `origin` instead.
PyAEDT WARNING: Argument `matname` is deprecated for method `create_cylinder`; use `material` instead.


In [5]:
box_object.subtract(cylinder_object, keep_originals=False)

PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec


In [6]:
cylinder_config = dict(
    cs_axis="Z" ,
    position=[0, 0, 0], 
    radius="$coax_radius", 
    height="$coax_height", 
    name="coax", 
    matname="copper"
)

coax_object = hfss.modeler.create_cylinder( **cylinder_config )

PyAEDT WARNING: Argument `cs_axis` is deprecated for method `create_cylinder`; use `orientation` instead.
PyAEDT WARNING: Argument `position` is deprecated for method `create_cylinder`; use `origin` instead.
PyAEDT WARNING: Argument `matname` is deprecated for method `create_cylinder`; use `material` instead.


In [7]:
box_object.unite(coax_object)

PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Union of 2 objects has been executed.


## Create antenna

In [8]:
if use_antenna:
    antenna_config = dict(
        cs_axis="Z" ,
        position=[0, 0, "$outer_height"], 
        radius="antenna_outer_radius", 
        height="$antenna_outer_height", 
        name="antenna", 
        matname="vacuum"
    )

    antenna_object = hfss.modeler.create_cylinder( **antenna_config )
    box_object.subtract(antenna_object, keep_originals=False)

    antenna_config = dict(
        cs_axis="Z" ,
        position=[0, 0, "$outer_height + $antenna_outer_height - $antenna_height"], 
        radius="antenna_radius", 
        height="$antenna_height", 
        name="core", 
        matname="copper"
    )

    core_object = hfss.modeler.create_cylinder( **antenna_config )

## Create chip

In [9]:
chip_coord = hfss.modeler.create_coordinate_system(origin = [0, "$chip_pos_y","$chip_pos_z"], name = "chip_coord")
chip_coord.set_as_working_cs()
cap1 = hfss.modeler.create_rectangle(origin = ["-0.5*$chip_width", "0.5*$chip_gap", 0], sizes = ["$chip_width", "$chip_height"], name = "cap1", orientation="XY")
cap2 = hfss.modeler.create_rectangle(origin = ["-0.5*$chip_width", "-0.5*$chip_gap - $chip_height", 0], sizes =  ["$chip_width", "$chip_height"], name = "cap2", orientation="XY")

In [10]:
## Only for newer pyaedt versions ?
hfss.assign_perfect_e("box")
hfss.assign_perfect_e("cap1")
hfss.assign_perfect_e("cap2")
if use_antenna:
    hfss.assign_perfect_e("core")

PyAEDT INFO: Boundary Perfect E PerfectE_box has been created.
PyAEDT INFO: Boundary Perfect E PerfectE_cap1 has been created.
PyAEDT INFO: Boundary Perfect E PerfectE_cap2 has been created.


## Create ports

In [11]:
# faces = vacuum_object.faces
# top_face = max(faces, key=lambda f: f.center[2])

In [12]:
hfss.modeler.set_working_coordinate_system("Global")
if solution_type=="Modal":
    if use_antenna:
        port_in = hfss.modeler.create_circle(origin = [0, 0, "$outer_height + $antenna_outer_height"], radius = "antenna_outer_radius", name = "port_in", orientation="XY")
        hfss.lumped_port(assignment="port_in", integration_line = hfss.AxisDir.YNeg, name="Port_in")
    else:
        port_in = hfss.modeler.create_circle(origin = [0, 0, "$outer_height"], radius = "$outer_radius", name = "port_in", orientation="XY")
        hfss.wave_port(assignment="port_in", name = "Port_in")

hfss.modeler.set_working_coordinate_system("chip_coord")
port_out = hfss.modeler.create_rectangle(origin = ["-10um","-0.5*$chip_gap" ], sizes=["20um","$chip_gap"], name = "port_out", orientation="XY")
if solution_type=="Eigenmode":
    hfss.assign_lumped_rlc_to_sheet(assignment="port_out", start_direction=hfss.AxisDir.YNeg, inductance=9e-9, name="Port_out")
else:
    hfss.lumped_port(assignment="port_out", integration_line = hfss.AxisDir.YNeg, name="Port_out")

PyAEDT INFO: Boundary Wave Port Port_in has been created.
PyAEDT INFO: Boundary Lumped Port Port_out has been created.


## Rotate chip

In [13]:
hfss.modeler.set_working_coordinate_system("chip_coord")
hfss.modeler.rotate(assignment=["cap1","cap2","port_out"], axis="X", angle="$chip_theta")

True

In [14]:
if use_second_chip:
    hfss.modeler.set_working_coordinate_system("Global")
    _, object_name = hfss.modeler.duplicate_around_axis(["cap1","cap2","port_out"], "Z", angle = 90)
    cap1_object = hfss.modeler.get_object_from_name(object_name[0])
    cap1_object.name = "chip2_cap1"
    cap2_object = hfss.modeler.get_object_from_name(object_name[1])
    cap2_object.name = "chip2_cap2"
    port_object = hfss.modeler.get_object_from_name(object_name[2])
    port_object.name = "chip2_port_out"

## Assign mesh operation

In [15]:
hfss.mesh.assign_length_mesh(["cap1", "cap2"], maximum_length="0.1mm")
hfss.mesh.assign_length_mesh(["port_out"], maximum_length="10um") # maximum 7um for JJ in qiskit-metal
if use_second_chip:
    hfss.mesh.assign_length_mesh(["chip2_cap1","chip2_cap2"], maximum_length="0.1mm")
    hfss.mesh.assign_length_mesh(["chip2_port_out"], maximum_length="10um")

PyAEDT INFO: Mesh class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Mesh class has been initialized! Elapsed time: 0m 0sec


## Create Analysis Setup

In [16]:
# if solution_type=="Modal":
#     hfss.create_open_region(Frequency="1GHz")

if solution_type=="Modal":

    setup = hfss.create_setup("MySetup") 
    setup.create_frequency_sweep(
        unit="GHz",
        name="Sweep1",
        start_frequency=3.0,
        stop_frequency=30.0,
        sweep_type="Interpolating",
    )

PyAEDT INFO: Parsing C:/HFSS/ymino/test.aedt.
PyAEDT INFO: File C:/HFSS/ymino/test.aedt correctly loaded. Elapsed time: 0m 0sec
PyAEDT INFO: aedt file load time 0.09594368934631348
PyAEDT INFO: Linear count sweep Sweep1 has been correctly created


In [17]:
if solution_type=="Eigenmode":
    setup = hfss.create_setup("MySetup") 
    setup.props["MinimumFrequency"] = "2GHz" 
    setup.props["NumModes"] = 3
    setup.props["MaximumPasses"] = 10
    print(setup.props)

In [18]:
if not (do_chip_angle_sweep or do_chip_gap_sweep or do_chip_width_sweep or do_chip_height_sweep or do_outer_radius_sweep or do_outer_height_sweep or do_coax_height_sweep):
    hfss.analyze_setup(name = "MySetup", cores=8)

In [19]:
if do_chip_angle_sweep:
    sweep = hfss.parametrics.add(
        variable="$chip_theta",
        start_point=0,
        end_point=90,
        step=5,
        name="ChipAngle"
    )
    hfss.set_oo_property_value(
        aedt_object=hfss.ooptimetrics, 
        object_name="ChipAngle", 
        prop_name='SaveFields', 
        value='True')
    sweep.analyze(cores = 8)

In [20]:
if do_chip_gap_sweep:
    sweep = hfss.parametrics.add(
        variable="$chip_gap",
        start_point=20,
        end_point=4000,
        step=10,
        name="ChipGap"
    )
    hfss.set_oo_property_value(
        aedt_object=hfss.ooptimetrics, 
        object_name="ChipGap", 
        prop_name='SaveFields', 
        value='True')
    sweep.analyze(cores = 8)

PyAEDT INFO: Key Desktop/ActiveDSOConfigurations/HFSS correctly changed.
PyAEDT INFO: Solving Optimetrics
PyAEDT INFO: Design setup ChipGap solved correctly in 0.0h 40.0m 39.0s


In [21]:
if do_chip_width_sweep:
    sweep = hfss.parametrics.add(
        variable="$chip_width",
        start_point=250,
        end_point=2000,
        step=5,
        name="ChipWidth"
    )
    hfss.set_oo_property_value(
        aedt_object=hfss.ooptimetrics, 
        object_name="ChipWidth", 
        prop_name='SaveFields', 
        value='True')
    sweep.analyze(cores = 8)

In [22]:
if do_chip_height_sweep:
    sweep = hfss.parametrics.add(
        variable="$chip_height",
        start_point=400,
        end_point=4000,
        step=5,
        name="ChipHeight"
    )
    hfss.set_oo_property_value(
        aedt_object=hfss.ooptimetrics, 
        object_name="ChipHeight", 
        prop_name='SaveFields', 
        value='True')
    sweep.analyze(cores = 8)

In [23]:
if do_outer_radius_sweep:
    radius_sweep = hfss.parametrics.add(
        variable="$outer_radius",
        start_point=5,
        end_point=15,
        step=6,
        name="Radius"
    )
    radius_sweep.analyze(cores = 8)

In [24]:
if do_outer_height_sweep:
    radius_sweep = hfss.parametrics.add(
        variable="$outer_height",
        start_point=18,
        end_point=38,
        step=11,
        name="OuterHeight"
    )
    hfss.set_oo_property_value(
        aedt_object=hfss.ooptimetrics, 
        object_name="OuterHeight", 
        prop_name='SaveFields', 
        value='True')
    radius_sweep.analyze(cores = 8)

In [25]:
if do_coax_height_sweep:
    height_sweep = hfss.parametrics.add(
        variable="$coax_height",
        start_point=9,
        end_point=38,
        step=3,
        name="Coax_height"
    )
    height_sweep.analyze(cores = 8)

## Produce Report

In [26]:
if solution_type == "Modal":
    report_config = dict(
        #expressions=["db(S12)"], 
        expressions=["db(S(Port_out,Port_in))","db(S(Port_out1,Port_in))"], 
        plot_name="S-parameter", 
        variations={
            "Freq": ["All"],
        }
    )
else:
    report_config = dict(
        expressions=["Mode(1)","Mode(2)","Mode(3)"], 
        plot_name="Eigen modes", 
        variations={}
    )

if do_chip_angle_sweep:
    report_config["variations"]["$chip_theta"] = ["All"] # ["0deg","90deg"]
if do_chip_gap_sweep:
    report_config["variations"]["$chip_gap"] = ["All"]
if do_chip_width_sweep:
    report_config["variations"]["$chip_width"] = ["All"]
if do_chip_height_sweep:
    report_config["variations"]["$chip_height"] = ["All"]
if do_outer_radius_sweep:
    report_config["variations"]["$outer_radius"] = ["All"]
if do_outer_height_sweep:
    report_config["variations"]["$outer_height"] = ["All"]
if do_coax_height_sweep:
    report_config["variations"]["$coax_height"] = ["All"]

report = hfss.post.create_report( **report_config )

PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Post class has been initialized! Elapsed time: 0m 0sec


In [27]:
# traces_to_plot = hfss.get_traces_for_plot()
# report = hfss.post.create_report(traces_to_plot)  # Creates a report in HFSS
# solution = report.get_solution_data()
# solution.plot(solution.expressions[1])  # Matplotlib axes object.

In [28]:
if solution_type == "Modal":
    expressions=["db(S(Port_out,Port_in))"]
    # report = hfss.post.reports_by_category.eigenmode(expressions=expressions)
    solution_data = report.get_solution_data()
    solution_data.export_data_to_csv(output="CoaxCavity_Modal.csv")
    print(solution_data.expressions)

PyAEDT INFO: Solution Data Correctly Loaded.
['db(S(Port_out,Port_in))']


In [29]:
if solution_type == "Eigenmode":
    expressions=["Mode(1)","Mode(2)","Mode(3)"]
    # report = hfss.post.reports_by_category.eigenmode(expressions=expressions)
    solution_data = report.get_solution_data()
    solution_data.export_data_to_csv(output="CoaxCavity.csv")
    solution_data.expressions
    for mode in expressions:
        print(mode)
        freq = solution_data.data_real(mode)[0]
        Q = abs(0.5 * freq / solution_data.data_imag(mode)[0])
        print("Frequency : ",freq*1e-9, " [GHz]")
        print("Q-value   : ",Q)

In [39]:
## Get intrinsic information (First you need the solution frequency)
intrinsics = hfss.setups[0].default_intrinsics
print(intrinsics)
print(hfss.setups)
solution = hfss.setups[0].get_solution_data()
# intrinsics["Freq"] = hfss.setups[0].properties["Solution Freq"]

{'Freq': '5GHz', 'Phase': '0deg'}
[MySetup with 2 Sweeps]
PyAEDT WARNING: Solution Data failed to load. Check solution, context or expression.
PyAEDT WARNING: No Data Available. Check inputs


In [31]:
## Plot field (Currently disabled)
# cutlist = ["Global:YZ"]
# setup_name = hfss.existing_analysis_sweeps[0]
# #setup_name = "Sweep1"
# quantity_name = "Mag_E"
# intrinsic = {"Freq": "5GHz", "Phase": "0deg"}
# hfss.logger.info("Generating the image")
# plot_obj = hfss.post.plot_field(
#         quantity="Mag_E",
#         assignment=cutlist,
#         plot_type="CutPlane",
#         setup=setup_name,
#         intrinsics=intrinsic
#     )

In [ ]:
#field_plot = hfss.post.create_fieldplot_volume( ["boundary"],"Mag_E", plot_name="boundary_Mag")
plane_plot = hfss.post.create_fieldplot_cutplane("Global:YZ", "Mag_E", plot_name = "plane_Mag")

PyAEDT INFO: Active Design set to A1


In [54]:
if solution_type=="Modal":
    # sweeps = hfss.get_sweeps("MySetup")
    # sweep = hfss.setups[0].sweeps[0]
    # print(sweep.frequencies)

    ooptimetric = hfss.get_oo_object(aedt_object=hfss.ooptimetrics, object_name="ChipGap")
    print(hfss.get_oo_properties(aedt_object=hfss.ooptimetrics, object_name="ChipGap"))
    print(hfss.get_oo_property_value(aedt_object=hfss.ooptimetrics, object_name="ChipGap", prop_name="IncludedVariables"))
    print(ooptimetric)

    # plane_plot.delete()
    # plane_plot = hfss.post.create_fieldplot_cutplane("Global:YZ", "Mag_E", plot_name = "plane_Mag", setup="MySetup: Sweep1")

['Name', 'Enabled', 'SaveFields', 'IncludedVariables', 'HasResult']
['$chip_gap']
Instance of an Aedt object:670


In [ ]:
if solution_type=="Eigenmode":
    sources = {"1": "1", "2": "0", "3": "0"}
    hfss.edit_sources(sources, eigenmode_stored_energy=True)

    min_value = hfss.post.get_scalar_field_value("Mag_E","Minimum")
    max_value = hfss.post.get_scalar_field_value("Mag_E","Maximum")
    print(max_value, min_value)

    plane_plot.change_plot_scale(maximum_value=max_value, minimum_value=1, is_log=True)
    # hfss.post.change_field_plot_scale(plot_name="plane_Mag1", maximum_value=max_value, minimum_value=1, is_log=True )

PyAEDT ERROR: **************************************************************
PyAEDT ERROR:   File "<frozen runpy>", line 198, in _run_module_as_main
PyAEDT ERROR:   File "<frozen runpy>", line 88, in _run_code
PyAEDT ERROR:   File "c:\Users\ymino\.conda\envs\pyaedt\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
PyAEDT ERROR:     app.launch_new_instance()
PyAEDT ERROR:   File "c:\Users\ymino\.conda\envs\pyaedt\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
PyAEDT ERROR:     app.start()
PyAEDT ERROR:   File "c:\Users\ymino\.conda\envs\pyaedt\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start
PyAEDT ERROR:     self.io_loop.start()
PyAEDT ERROR:   File "c:\Users\ymino\.conda\envs\pyaedt\Lib\site-packages\tornado\platform\asyncio.py", line 211, in start
PyAEDT ERROR:     self.asyncio_loop.run_forever()
PyAEDT ERROR:   File "c:\Users\ymino\.conda\envs\pyaedt\Lib\asyncio\base_events.py", line 608, in run_forever
PyAEDT ERROR:     s

True

In [35]:
## The following function is correct, but doesn't work due to PyVista...
# hfss.post.plot_field(
#     quantity="Mag_E",
#     assignment="Global:YZ",
#     plot_type="CutPlane",
#     log_scale=True,
# )